In [1]:
%load_ext autoreload
%autoreload 2

In [2]:
import sys
import os
import torch
import copy
from torch.utils.data import DataLoader, Subset
import torch.nn as nn
import matplotlib.pyplot as plt
import numpy as np
from collections import defaultdict
from tqdm import tqdm

project_root = os.path.abspath(os.path.join(os.getcwd(), ".."))
if project_root not in sys.path:
    sys.path.append(project_root)

In [3]:
from src.trainer import SimpleTrainer, FisherTrainer, IntervalTrainer
from src.data_utils import (
    get_mnist_tasks,
    _extract_targets,
    get_context_sets,
    create_holdout_set,
)
from src.utils.general import InContextHead
from src import models
from src.regulariser import UnbiasRegulariser, L2Regulariser, MultiRegulariser

from configs import MNIST_FISHER_CONFIG as CONFIG

In [ ]:
SEED = 0
train_tasks, val_tasks, test_tasks = get_mnist_tasks(seed=SEED)

context_sets = get_context_sets(test_tasks)
head = InContextHead(context_sets, 10, device="cuda")
head.set_context(0)
model = models.get_mnist_model(head=head, device="cuda", seed=SEED)

print(
    f"Tasks: {[torch._unique(_extract_targets(train))[0].tolist() for train in train_tasks]}"
)


In [ ]:
interval_trainer = IntervalTrainer(
    model,
    checkpoint=CONFIG["checkpoint"],
    n_iters=CONFIG["n_iters"],
    min_acc_limit=CONFIG["min_acc_limit"],
    min_acc_increment=CONFIG["min_acc_increment"],
    primal_learning_rate=CONFIG["primal_learning_rate"],
    dual_learning_rate=CONFIG["dual_learning_rate"],
    projection_strategy=CONFIG["projection_strategy"],
    n_certificate_samples=CONFIG["n_certificate_samples"],
    penalty_coefficient=CONFIG["penalty_coefficient"],
    paradigm="TIL",
    seed=SEED,
)

interval_trainer.train(
    train_tasks[0],
    val_tasks[0],
    batch_size=CONFIG["batch_size"],
    epochs=CONFIG["epochs"],
    lr=CONFIG["lr"],
    weight_decay=CONFIG["weight_decay"],
    context_id=0,
)

interval_trainer.compute_rashomon_set(test_tasks[0], context_id=0)

interval_trainer.train(
    train_tasks[1],
    val_tasks[1],
    batch_size=CONFIG["batch_size"],
    epochs=CONFIG["epochs"],
    lr=CONFIG["lr"],
    weight_decay=CONFIG["weight_decay"],
    context_id=1,
)
interval_trainer.test(test_tasks, context_list=list(range(len(test_tasks))))

In [ ]:
fisher_trainer = FisherTrainer(
    model,
    checkpoint=CONFIG["checkpoint"],
    n_iters=CONFIG["n_iters"],
    min_acc_limit=CONFIG["min_acc_limit"],
    min_acc_increment=CONFIG["min_acc_increment"],
    primal_learning_rate=CONFIG["primal_learning_rate"],
    dual_learning_rate=CONFIG["dual_learning_rate"],
    projection_strategy=CONFIG["projection_strategy"],
    n_certificate_samples=CONFIG["n_certificate_samples"],
    penalty_coefficient=CONFIG["penalty_coefficient"],
    paradigm="TIL",
    seed=SEED,
)

fisher_trainer.train(
    train_tasks[0],
    val_tasks[0],
    batch_size=CONFIG["batch_size"],
    epochs=CONFIG["epochs"],
    lr=CONFIG["lr"],
    weight_decay=CONFIG["weight_decay"],
    context_id=0,
)

fisher_trainer.compute_rashomon_set(
    test_tasks[0], test_tasks[1], context_id=0, prune_prop=0
)

fisher_trainer.train(
    train_tasks[1],
    val_tasks[1],
    batch_size=CONFIG["batch_size"],
    epochs=CONFIG["epochs"],
    lr=CONFIG["lr"],
    weight_decay=CONFIG["weight_decay"],
    context_id=1,
)
fisher_trainer.test(test_tasks, context_list=list(range(len(test_tasks))))

### Task Incremental Learning

In [ ]:
SEED = 0
train_tasks, val_tasks, test_tasks = get_mnist_tasks(seed=SEED)

context_sets = get_context_sets(test_tasks)
head = InContextHead(context_sets, 10, device="cuda")
head.set_context(0)
model = models.get_mnist_model(head=head, device="cuda", seed=SEED)

print(
    f"Tasks: {[torch._unique(_extract_targets(train))[0].tolist() for train in train_tasks]}"
)

In [ ]:
fisher_trainer = FisherTrainer(
    model,
    checkpoint=CONFIG["checkpoint"],
    n_iters=CONFIG["n_iters"],
    min_acc_limit=CONFIG["min_acc_limit"],
    min_acc_increment=CONFIG["min_acc_increment"],
    primal_learning_rate=CONFIG["primal_learning_rate"],
    dual_learning_rate=CONFIG["dual_learning_rate"],
    projection_strategy=CONFIG["projection_strategy"],
    n_certificate_samples=CONFIG["n_certificate_samples"],
    penalty_coefficient=CONFIG["penalty_coefficient"],
    paradigm="TIL",
    seed=SEED,
)

for i, (train, val, test) in enumerate(zip(train_tasks, val_tasks, test_tasks)):
    fisher_trainer.train(
        train,
        val,
        batch_size=CONFIG["batch_size"],
        epochs=CONFIG["epochs"],
        lr=CONFIG["lr"],
        weight_decay=CONFIG["weight_decay"],
        context_id=i,
    )
    fisher_trainer.test(test_tasks, context_list=list(range(len(test_tasks))))

    if i < len(test_tasks) - 1:
        fisher_trainer.compute_rashomon_set(
            test,
            test_tasks[i + 1],
            context_id=i,
            prune_prop=CONFIG["prune_prop"],
            fisher_batch_size=CONFIG["fisher_batch_size"],
            fisher_epochs=CONFIG["fisher_epochs"],
        )

### Domain Incremental Learning

In [4]:
SEED = 0
train_tasks, val_tasks, test_tasks = get_mnist_tasks(seed=SEED)

model = models.get_mnist_model(device="cuda", output_dim=2, seed=SEED)

print(
    f"Tasks: {[torch._unique(_extract_targets(train))[0].tolist() for train in train_tasks]}"
)

Tasks: [[7, 8], [0, 9], [1, 2], [3, 6], [4, 5]]


In [5]:
def domain_map_fn(labels: torch.Tensor) -> torch.Tensor:
    """Map the global label to the in context label."""
    return labels % 2

In [7]:
fisher_trainer = FisherTrainer(
    model,
    checkpoint=CONFIG["checkpoint"],
    n_iters=CONFIG["n_iters"],
    min_acc_limit=CONFIG["min_acc_limit"],
    min_acc_increment=CONFIG["min_acc_increment"],
    primal_learning_rate=CONFIG["primal_learning_rate"],
    dual_learning_rate=CONFIG["dual_learning_rate"],
    projection_strategy=CONFIG["projection_strategy"],
    n_certificate_samples=CONFIG["n_certificate_samples"],
    penalty_coefficient=CONFIG["penalty_coefficient"],
    paradigm="DIL",
    domain_map_fn=domain_map_fn,
    seed=SEED,
)

for i, (train, val, test) in enumerate(zip(train_tasks, val_tasks, test_tasks)):
    fisher_trainer.train(
        train,
        val,
        batch_size=CONFIG["batch_size"],
        epochs=CONFIG["epochs"],
        lr=CONFIG["lr"],
        weight_decay=CONFIG["weight_decay"],
    )
    fisher_trainer.test(test_tasks)

    if i < len(test_tasks) - 1:
        fisher_trainer.compute_rashomon_set(
            test,
            test_tasks[i + 1],
            prune_prop=CONFIG["prune_prop"],
            fisher_batch_size=CONFIG["fisher_batch_size"],
            fisher_epochs=CONFIG["fisher_epochs"],
        )

Training Epochs: 100%|█████████████████████████| 5/5 [00:05<00:00,  1.10s/it, val_loss=0.0110, val_acc=0.9967, proj=None]


Test Results: [(0.0235, 0.9915), (0.5784, 0.8381), (1.2203, 0.5708), (3.1147, 0.4365), (5.4571, 0.1852)]
To keep top 20%, found global Fisher threshold: 0.5763
---------------------------- Computing Rashomon set ----------------------------
Initial acc constraint violation: -0.1250 (Positive = violated)
Number of model parameters: 1563
Computing Rashomon set with min acc limit: 0.87
Initial bbox:  Obj=0.00,  Size=0.00,  Min acc hard=0.99,  Min acc soft=0.99


100%|███████████████████████████████████████| 200/200 [00:08<00:00, 23.71it/s, size=20.32, obj=0.013, min_soft_acc=1.000]


Final bbox:  Obj=0.01,  Size=20.32,  Min acc hard=0.86,  Min acc soft=0.86
Computing final certificates over 400 samples
Checkpointed every 20 iterations for a total of 10 checkpoints
Checkpoints sizes: ['2.96', '9.93', '14.39', '17.52', '18.03', '18.71', '19.35', '19.71', '20.20', '20.32']
Checkpoint certificates: ['0.93', '0.85', '0.87', '0.86', '0.86', '0.85', '0.86', '0.87', '0.86', '0.86']
----------------------- Finished Computing Rashomon set ------------------------


Training Epochs: 100%|████████████████████████████| 5/5 [00:10<00:00,  2.11s/it, val_loss=0.2743, val_acc=0.9280, proj=8]


...

Test Results: [(0.0351, 0.9855), (0.2674, 0.9271), (1.3399, 0.5778), (3.5894, 0.4842), (7.2028, 0.1153)]
To keep top 20%, found global Fisher threshold: 2.6909
---------------------------- Computing Rashomon set ----------------------------
Initial acc constraint violation: -0.0692 (Positive = violated)
Computing Rashomon set within outer box of size: 0.27


Training Epochs: 100%|████████████████████████████| 5/5 [00:06<00:00,  1.25s/it, val_loss=3.3259, val_acc=0.5185, proj=0]


Test Results: [(0.0362, 0.984), (0.2673, 0.9251), (1.4002, 0.5671), (3.5798, 0.4929), (7.2106, 0.1185)]
To keep top 20%, found global Fisher threshold: 3.9745
---------------------------- Computing Rashomon set ----------------------------
Initial acc constraint violation: -0.1062 (Positive = violated)
Computing Rashomon set within outer box of size: 0.27


Training Epochs: 100%|████████████████████████████| 5/5 [00:05<00:00,  1.17s/it, val_loss=6.5623, val_acc=0.1269, proj=0]


Test Results: [(0.0351, 0.9855), (0.2674, 0.9271), (1.3399, 0.5778), (3.5894, 0.4842), (7.2028, 0.1153)]


### Class Incremental Learning

In [4]:
SEED = 0
train_tasks, val_tasks, test_tasks = get_mnist_tasks(seed=SEED)

model = models.get_mnist_model(device="cuda", output_dim=10, seed=SEED)

print(
    f"Tasks: {[torch._unique(_extract_targets(train))[0].tolist() for train in train_tasks]}"
)

Tasks: [[7, 8], [0, 9], [1, 2], [3, 6], [4, 5]]


In [9]:
fisher_trainer = FisherTrainer(
    model,
    checkpoint=CONFIG["checkpoint"],
    n_iters=CONFIG["n_iters"],
    min_acc_limit=CONFIG["min_acc_limit"],
    min_acc_increment=CONFIG["min_acc_increment"],
    primal_learning_rate=CONFIG["primal_learning_rate"],
    dual_learning_rate=CONFIG["dual_learning_rate"],
    projection_strategy=CONFIG["projection_strategy"],
    n_certificate_samples=CONFIG["n_certificate_samples"],
    penalty_coefficient=CONFIG["penalty_coefficient"],
    paradigm="CIL",
    seed=SEED,
)

for i, (train, val, test) in enumerate(zip(train_tasks, val_tasks, test_tasks)):
    fisher_trainer.train(
        train,
        val,
        batch_size=CONFIG["batch_size"],
        epochs=CONFIG["epochs"],
        lr=CONFIG["lr"],
        weight_decay=CONFIG["weight_decay"],
    )
    fisher_trainer.test(test_tasks)

    if i < len(test_tasks) - 1:
        fisher_trainer.compute_rashomon_set(
            test,
            test_tasks[i + 1],
            prune_prop=CONFIG["prune_prop"],
            fisher_batch_size=CONFIG["fisher_batch_size"],
            fisher_epochs=CONFIG["fisher_epochs"],
        )

Training Epochs: 100%|█████████████████████████| 5/5 [00:05<00:00,  1.15s/it, val_loss=0.0141, val_acc=0.9947, proj=None]


Test Results: [(0.0358, 0.9865), (46.3004, 0.0), (39.2716, 0.0), (47.2765, 0.0), (43.7228, 0.0)]
To keep top 20%, found global Fisher threshold: 3.4296
---------------------------- Computing Rashomon set ----------------------------
Initial acc constraint violation: -0.1251 (Positive = violated)
Number of model parameters: 6179
Computing Rashomon set with min acc limit: 0.86
Initial bbox:  Obj=0.00,  Size=0.00,  Min acc hard=0.99,  Min acc soft=0.99


100%|██████████████████████████████████████| 200/200 [00:08<00:00, 24.00it/s, size=112.70, obj=0.018, min_soft_acc=1.000]


Final bbox:  Obj=0.02,  Size=112.70,  Min acc hard=0.93,  Min acc soft=0.92
Computing final certificates over 400 samples
Checkpointed every 20 iterations for a total of 10 checkpoints
Checkpoints sizes: ['4.07', '16.64', '36.25', '55.06', '64.26', '74.36', '85.02', '94.96', '103.73', '112.70']
Checkpoint certificates: ['0.92', '0.94', '0.95', '0.96', '0.94', '0.96', '0.87', '0.88', '0.88', '0.93']
----------------------- Finished Computing Rashomon set ------------------------


Training Epochs: 100%|████████████████████████████| 5/5 [00:10<00:00,  2.13s/it, val_loss=2.9010, val_acc=0.4888, proj=9]


...

Test Results: [(0.0427, 0.9845), (2.691, 0.4615), (39.1152, 0.0), (47.2319, 0.0), (43.4716, 0.0)]
To keep top 20%, found global Fisher threshold: 2.2036
---------------------------- Computing Rashomon set ----------------------------


Training Epochs: 100%|███████████████████████████| 5/5 [00:06<00:00,  1.26s/it, val_loss=46.3311, val_acc=0.0000, proj=0]


Test Results: [(0.0427, 0.9845), (2.691, 0.4615), (39.1152, 0.0), (47.2319, 0.0), (43.4716, 0.0)]
To keep top 20%, found global Fisher threshold: 2.6168
---------------------------- Computing Rashomon set ----------------------------


Training Epochs: 100%|███████████████████████████| 5/5 [00:05<00:00,  1.17s/it, val_loss=42.5289, val_acc=0.0000, proj=0]


Test Results: [(0.0427, 0.9845), (2.691, 0.4615), (39.1152, 0.0), (47.2319, 0.0), (43.4716, 0.0)]


### Class Incremental Learning + Regulariser

In [5]:
SEED = 1
train_tasks, val_tasks, test_tasks = get_mnist_tasks(seed=SEED)

model = models.get_mnist_model(device="cuda", output_dim=10, seed=SEED)

print(
    f"Tasks: {[torch._unique(_extract_targets(train))[0].tolist() for train in train_tasks]}"
)

Tasks: [[0, 7], [5, 8], [4, 9], [3, 6], [1, 2]]


In [6]:
unbias = UnbiasRegulariser(
    lmbd=CONFIG["unbias_lambda"],
    unbias_domain=[
        torch.zeros(1, 1, 28, 28, device="cuda"),
        torch.ones(1, 1, 28, 28, device="cuda"),
    ],
)
l2 = L2Regulariser(lmbd=CONFIG["l2_lambda"])
regulariser = MultiRegulariser([l2, unbias])

In [8]:
fisher_trainer = FisherTrainer(
    model,
    checkpoint=CONFIG["checkpoint"],
    n_iters=CONFIG["n_iters"],
    min_acc_limit=CONFIG["min_acc_limit"],
    min_acc_increment=CONFIG["min_acc_increment"],
    primal_learning_rate=CONFIG["primal_learning_rate"],
    dual_learning_rate=CONFIG["dual_learning_rate"],
    projection_strategy=CONFIG["projection_strategy"],
    n_certificate_samples=CONFIG["n_certificate_samples"],
    penalty_coefficient=CONFIG["penalty_coefficient"],
    paradigm="CIL",
    seed=SEED,
)

for i, (train, val, test) in enumerate(zip(train_tasks, val_tasks, test_tasks)):
    fisher_trainer.train(
        train,
        val,
        batch_size=CONFIG["batch_size"],
        epochs=CONFIG["epochs"],
        lr=CONFIG["lr"],
        weight_decay=CONFIG["weight_decay"],
        regulariser=regulariser
    )
    fisher_trainer.test(test_tasks)

    if i < len(test_tasks) - 1:
        fisher_trainer.compute_rashomon_set(
            test,
            test_tasks[i + 1],
            prune_prop=CONFIG["prune_prop"],
            fisher_batch_size=CONFIG["fisher_batch_size"],
            fisher_epochs=CONFIG["fisher_epochs"],
        )

Training Epochs: 100%|█████████████████████████| 5/5 [00:07<00:00,  1.55s/it, val_loss=0.0172, val_acc=0.9960, proj=None]


Test Results: [(0.0154, 0.996), (8.371, 0.0), (8.7936, 0.0), (8.9122, 0.0), (7.1308, 0.0)]
To keep top 20%, found global Fisher threshold: 0.3480
---------------------------- Computing Rashomon set ----------------------------
Initial acc constraint violation: -0.1177 (Positive = violated)
Number of model parameters: 6179
Computing Rashomon set with min acc limit: 0.88
Initial bbox:  Obj=0.00,  Size=0.00,  Min acc hard=1.00,  Min acc soft=1.00


100%|██████████████████████████████████████| 400/400 [00:17<00:00, 23.53it/s, size=269.28, obj=0.043, min_soft_acc=0.875]


Final bbox:  Obj=0.04,  Size=269.28,  Min acc hard=0.88,  Min acc soft=0.88
Computing final certificates over 400 samples
Checkpointed every 20 iterations for a total of 20 checkpoints
Checkpoints sizes: ['7.56', '20.43', '42.85', '55.49', '72.26', '87.27', '98.84', '114.15', '128.16', '141.01', '153.16', '166.20', '180.07', '193.79', '205.85', '217.83', '230.40', '243.98', '256.07', '269.28']
Checkpoint certificates: ['0.94', '0.98', '0.94', '0.95', '0.87', '0.88', '0.92', '0.87', '0.87', '0.89', '0.88', '0.89', '0.89', '0.89', '0.88', '0.89', '0.90', '0.88', '0.88', '0.88']
----------------------- Finished Computing Rashomon set ------------------------


Training Epochs: 100%|███████████████████████████| 5/5 [00:15<00:00,  3.03s/it, val_loss=0.5489, val_acc=0.8596, proj=19]


...

Test Results: [(0.2038, 0.9671), (0.5486, 0.8526), (9.5226, 0.0), (9.8028, 0.0), (8.1784, 0.0)]
To keep top 20%, found global Fisher threshold: 0.2495
---------------------------- Computing Rashomon set ----------------------------


Training Epochs: 100%|████████████████████████████| 5/5 [00:07<00:00,  1.53s/it, val_loss=9.6596, val_acc=0.0000, proj=0]


Test Results: [(0.2038, 0.9671), (0.5486, 0.8526), (9.5226, 0.0), (9.8028, 0.0), (8.1784, 0.0)]
To keep top 20%, found global Fisher threshold: 0.1648
---------------------------- Computing Rashomon set ----------------------------


Training Epochs: 100%|████████████████████████████| 5/5 [00:08<00:00,  1.61s/it, val_loss=8.3019, val_acc=0.0000, proj=0]


Test Results: [(0.2038, 0.9671), (0.5486, 0.8526), (9.5226, 0.0), (9.8028, 0.0), (8.1784, 0.0)]


### Experiments

In [ ]:
def run_experiment(seed, require_prev=True, pruning_prop=0.7):
    train_tasks, val_tasks, test_tasks = get_mnist_tasks(seed=seed)

    model = models.get_mnist_model(seed=seed)
    model.to("mps")

    train_tasks, _ = zip(*[create_holdout_set(dataset) for dataset in train_tasks])
    trainer = SimpleTrainer(model, seed=seed)
    trainer.train(train_tasks[0], val_tasks[0], epochs=3, batch_size=256)
    trainer.test(test_tasks[0:1])

    save_model = copy.deepcopy(trainer.model)

    interval_trainer = IntervalTrainer(
        save_model,
        checkpoint=200,
        n_iters=200,
        min_acc_limit=0.8,
        min_acc_increment=0,
        primal_learning_rate=0.5,
        paradigm="CIL",
        seed=seed,
    )

    # Compute bounds for task 0
    prev = []
    if require_prev:
        interval_trainer.compute_rashomon_set(test_tasks[0])

        assert interval_trainer.test(test_tasks[0:2])[1][1] == 0, (
            "Base model performance on second task needs to be zero."
        )
        # Train task 1 until plateau
        interval_trainer.train(train_tasks[1], val_tasks[1], batch_size=256, epochs=3)
        print("No pruning performance: ", end="")
        prev = interval_trainer.test(test_tasks[0:2])

    fisher_trainer = FisherTrainer(
        save_model,
        checkpoint=200,
        n_iters=200,
        min_acc_limit=0.8,
        min_acc_increment=0,
        primal_learning_rate=0.5,
        paradigm="CIL",
        seed=seed,
    )

    loader = DataLoader(
        Subset(train_tasks[1], range(64)),
        batch_size=64,
        shuffle=True,
        generator=torch.Generator().manual_seed(seed),
    )

    fisher_trainer._compute_fisher_diagonal(loader)

    fisher_trainer.compute_rashomon_set(test_tasks[0], prune_prop=pruning_prop)
    assert fisher_trainer.test(test_tasks[0:2])[1][1] == 0
    fisher_trainer.train(train_tasks[1], val_tasks[1], epochs=3, batch_size=256)
    new = fisher_trainer.test(test_tasks[0:2])

    return prev, new

In [ ]:
prev_perf = []
new_perf = []
# reds = []
for i in range(10):
    result = run_experiment(i, pruning_prop=0.8)
    prev_perf.append(result[0])
    new_perf.append(result[1])
    # reds.append(result[2])

In [ ]:
prev_acc = np.array([val[1][1] for val in prev_perf]).mean()
prev_std = np.array([val[1][1] for val in prev_perf]).std()
new_acc = np.array([val[1][1] for val in new_perf]).mean()
new_std = np.array([val[1][1] for val in new_perf]).std()

# 2. Create a prettier plot
# Use a more balanced figure size
fig, ax = plt.subplots(figsize=(7, 6))

# Define some nice colors
colors = ["#4682B4", "#5FBA7D"]  # Steel Blue and a nice Green

# Plot the bars with improved styling
bars = ax.bar(
    x=["Previous", "New"],
    height=[prev_acc, new_acc],
    yerr=[prev_std, new_std],
    color=colors,
    alpha=0.8,  # Make bars slightly transparent
    edgecolor="black",  # Add a crisp black edge
    capsize=10,  # THIS IS KEY: Adds caps to the error bars
    ecolor="black",  # Color of the error bar lines
    linewidth=1.5,
)

# 3. Add Labels, Title, and Grid for context
ax.set_ylabel("Accuracy", fontsize=12)
ax.set_title("Comparison of Previous vs. New Performance", fontsize=16, pad=20)
ax.set_xticks(ticks=[0, 1], labels=["No lookahead", "50 step lookahead"], fontsize=12)

# Add a subtle horizontal grid to make comparisons easier
ax.yaxis.grid(True, linestyle="--", which="major", color="grey", alpha=0.3)

# Remove the top and right spines for a cleaner look
ax.spines["top"].set_visible(False)
ax.spines["right"].set_visible(False)

# Set a dynamic Y-axis limit for better spacing
ax.set_ylim(0, max(new_acc, prev_acc) + max(new_std, prev_std) + 0.05)

# 4. Add data labels on top of the bars
# This shows the exact mean value on the plot
ax.bar_label(bars, fmt="{:.3f}", padding=5, fontsize=11, color="black")

# Ensure everything fits without overlapping
plt.tight_layout()
plt.show()

In [ ]:
data = defaultdict(list)
pruning_prop = [0, 0.05, 0.1, 0.3, 0.5, 0.7, 0.8, 0.85]
for prop in tqdm(pruning_prop):
    for i in range(3):
        data[prop].append(run_experiment(i, pruning_prop=prop))

In [ ]:
x_vals, y_means, y_stds = [], [], []
prev_means = []
prop_mean = []
for key, item in data.items():
    x_vals.append(key)
    # Create the numpy array once
    values = np.array([val[1][1][1] for val in item])
    y_means.append(values.mean())
    y_stds.append(values.std())
    prop_mean.append(np.array([val[0] for val in item]).mean())
    prev_means.append(np.array([val[0][1][1] for val in item]).mean())

# Convert lists to numpy arrays for easier calculations
x_vals = np.array(x_vals)
y_means = np.array(y_means)
y_stds = np.array(y_stds)

# 2. Create a prettier plot
# Use a pre-defined style for a professional look
plt.style.use("seaborn-v0_8-whitegrid")
fig, ax = plt.subplots(figsize=(9, 6))

# Define a color for the plot
primary_color, secondary_color = ["#4682B4", "#5FBA7D"]

# 3. Plot the main line with markers
ax.plot(
    x_vals,
    y_means,
    color=primary_color,
    linewidth=2.5,
    marker="o",  # Add circles at each data point
    markersize=8,
    label="Mean Performance",
)

ax.plot(
    x_vals,
    prev_means,
    color=secondary_color,
    linewidth=2.5,
    markersize=8,
    linestyle="--",
    label="Mean Baseline Performance",
)

# 4. Add the shaded error band (the key improvement!)
# The alpha value makes the shade transparent
ax.fill_between(
    x_vals,
    y_means - y_stds,  # Lower bound of the error
    y_means + y_stds,  # Upper bound of the error
    color=primary_color,
    alpha=0.2,
    label="Standard Deviation",
)

# 5. Add Labels, Title, and Legend
ax.set_title("Performance vs. Pruning Level", fontsize=16, pad=20)
ax.set_xlabel("Pruning Proportion", fontsize=12)
ax.set_ylabel("Accuracy", fontsize=12)
ax.legend(loc="upper right", fontsize=11)

# Customize tick parameters
ax.tick_params(axis="both", which="major", labelsize=11)

# plt.plot(x_vals, prop_mean)

# Ensure everything fits nicely
plt.tight_layout()
plt.show()
